# Plant disease classification under dataset shift

## BaselineCNN vs. ResNet-18: PlantVillage to PlantDoc

This project compares a **custom CNN (BaselineCNN)** trained from scratch to an **ImageNet-pretrained ResNet-18** on the same eight disease classes. **PlantVillage** is used for **training**, **validation-only tuning and model selection**, and **in-domain test** evaluation. **PlantDoc** is used **only after** model selection, as a **final external** generalization and dataset-shift test.

Large drops in accuracy on PlantDoc are **consistent with** strong **domain shift** and are **compatible with** **possible** shortcut learning (e.g. reliance on non-disease spurious cues). This notebook does **not** claim proof of background-only learning or a single spurious feature per error.

---

### Google Colab (optional)

Clone the repository and set your working directory to the **project root** (or run from `notebooks/`). The first code cell **auto-detects** the repository root in either case.

```python
!git clone https://github.com/Kyrie21323/Plant-Disease-Classification.git
%cd Plant-Disease-Classification
```


In [ ]:
# Imports, paths, and setup
import json
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display  # type: ignore[import]

def _find_repo_root() -> Path:
    """Project root: directory containing both configs/class_subset_v1.json and src/."""
    marker = Path("configs") / "class_subset_v1.json"
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        if (base / marker).is_file() and (base / "src").is_dir():
            return base
    p = cwd
    for _ in range(20):
        p = p.parent
        if (p / marker).is_file() and (p / "src").is_dir():
            return p
        if p == p.parent:
            break
    raise FileNotFoundError(
        "Could not find project root (need configs/class_subset_v1.json and src/). "
        "cd into the cloned repository root (e.g. after %cd Plant-Disease-Classification) "
        "or the notebooks/ folder, then re-run this cell."
    )

REPO_ROOT = _find_repo_root()
print(f"REPO_ROOT = {REPO_ROOT}")
RESULTS = REPO_ROOT / "outputs" / "results"
FIGURES = REPO_ROOT / "outputs" / "figures"
CONFIG = REPO_ROOT / "configs" / "class_subset_v1.json"
DATA_SPLITS = REPO_ROOT / "data" / "splits"
DATA_META = REPO_ROOT / "data" / "metadata"
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def set_seed(s: int = 42) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed(42)

with open(CONFIG, encoding="utf-8") as f:
    CLASS_SUBSET = json.load(f)

EXP_PV_TOTAL, EXP_PD_TOTAL = 11_819, 940
EXP_TRAIN, EXP_VAL, EXP_TEST = 8273, 1773, 1773

def load_json_path(path: Path):
    if not path.exists():
        warnings.warn(f"Missing file: {path}")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def show_figure(rel: str, width: int = 720) -> None:
    p = (REPO_ROOT / rel).resolve()
    if not p.exists():
        print(f"WARNING: figure not found: {p}")
        return
    display(Image(str(p), width=width))

## Final selected 8-class subset (V1)

Classes have **unambiguous** PlantVillage↔PlantDoc name pairs, **disease** (not noisy healthy) labels, **sufficient** PlantDoc images, and **multiple plant species** (corn, tomato, squash, potato). See `docs/CLASS_MAPPING.md` and `docs/FINAL_CLASS_SUBSET.md`.

In [ ]:
rows = [
    {
        "label_id": d["label_id"],
        "unified_label": d["unified_label"],
        "plant": d["plant"],
        "disease": d["disease"],
        "plantvillage_class": d["plantvillage_class"],
        "plantdoc_class": d["plantdoc_class"],
    }
    for d in sorted(CLASS_SUBSET, key=lambda x: x["label_id"])
]
df_classes = pd.DataFrame(rows)
display(df_classes)

## Dataset metadata and experimental protocol

- **PlantVillage train** → supervised **training** for both models.
- **PlantVillage validation** → **only** **hyperparameter tuning and model selection** (Baseline runs).
- **PlantVillage test** → **final in-domain** evaluation after selection.
- **PlantDoc** → **final external** generalization and dataset-shift analysis **only**; **not** used for tuning, checkpoint selection, or relabeling.

**Expected subset sizes (V1):** 11,819 PlantVillage and 940 PlantDoc images; **expected split** 8273 / 1773 / 1773 (train / val / test) when CSVs are present.

In [ ]:
def count_csv_rows(p: Path):
    if not p.exists():
        return None
    return len(pd.read_csv(p))

pv_train_csv = DATA_SPLITS / "plantvillage_train_split.csv"
pv_val_csv = DATA_SPLITS / "plantvillage_val_split.csv"
pv_test_csv = DATA_SPLITS / "plantvillage_test_split.csv"
pd_meta_csv = DATA_META / "plantdoc_subset_metadata.csv"

n_train, n_val, n_test = (
    count_csv_rows(pv_train_csv),
    count_csv_rows(pv_val_csv),
    count_csv_rows(pv_test_csv),
)
n_pd = count_csv_rows(pd_meta_csv) if pd_meta_csv.exists() else None

if n_train and n_val and n_test:
    n_pv = n_train + n_val + n_test
    print(f"From CSVs: PlantVillage train/val/test = {n_train} / {n_val} / {n_test}  (total {n_pv})")
else:
    n_train, n_val, n_test, n_pv = EXP_TRAIN, EXP_VAL, EXP_TEST, EXP_PV_TOTAL
    print("Split CSVs not all found; using documented sizes:")
    print(f"  train/val/test = {n_train} / {n_val} / {n_test}  (total {n_pv})")

if n_pd is None:
    n_pd = EXP_PD_TOTAL
    print(f"PlantDoc metadata CSV not found; documented PlantDoc subset total = {n_pd}")
else:
    print(f"PlantDoc metadata rows: {n_pd}")

## EDA and visual dataset comparison

Below: **class distribution** and **sample grids** (saved in `outputs/figures/`). PlantVillage is relatively **lab-style** (cleaner context); PlantDoc is more **web/field** style with busier **backgrounds**, **lighting**, and **framing**. These differences are **suggestive** of **strong domain shift** and help explain a large **in-domain → external** performance gap.

In [ ]:
for rel in [
    "outputs/figures/final_subset_class_distribution.png",
    "outputs/figures/plantvillage_sample_grid.png",
    "outputs/figures/plantdoc_sample_grid.png",
]:
    display(Markdown(f"`{rel}`"))
    show_figure(rel)

## Preprocessing and transforms

- **Train:** `get_train_transform()` (random crop, flip, rotation, **mild** ColorJitter in the “selected” setting).
- **Val / test / PlantDoc eval:** `get_eval_transform()` — **deterministic** resize+normalize, **no** random augmentations.
- **ImageNet** mean/std in `data/transforms.py` so **BaselineCNN** and **ResNet-18** get **comparable** tensors.
- **Why shared eval?** The **same** eval pipeline on **PlantVillage test** and **PlantDoc** is required so the generalization gap reflects **data shift + model**, not different preprocessing.

In [ ]:
from data.transforms import IMAGE_SIZE, NORMALIZE_MEAN, NORMALIZE_STD, get_eval_transform, get_train_transform

print("IMAGE_SIZE:", IMAGE_SIZE)
print("ImageNet mean/std:", NORMALIZE_MEAN, NORMALIZE_STD)
print("get_train_transform():", get_train_transform())
print("get_eval_transform():", get_eval_transform())

## DataLoaders

We call `build_dataloaders()` with **paths under this clone** of the repo (not the hardcoded WSL string inside `dataloaders.py`). This **one** call builds PV train/val/test + PlantDoc. **If CSVs are missing** (e.g. data not in the working tree), we skip the demo batch.

In [ ]:
from data.dataloaders import build_dataloaders

paths_ok = all(
    p.exists() for p in (pv_train_csv, pv_val_csv, pv_test_csv, pd_meta_csv)
)
if not paths_ok:
    print(
        "WARNING: Missing one or more split/metadata CSVs; skip DataLoader build. "
        "Add data under data/splits/ and data/metadata/ to run this block."
    )
else:
    _ds, loaders = build_dataloaders(
        batch_size=32,
        num_workers=0,
        pin_memory=False,
        train_csv=pv_train_csv,
        val_csv=pv_val_csv,
        test_csv=pv_test_csv,
        pd_csv=pd_meta_csv,
    )
    print(
        "Dataset sizes: train =", len(_ds.pv_train),
        "val =", len(_ds.pv_val),
        "test =", len(_ds.pv_test),
        "PlantDoc =", len(_ds.pd_eval),
    )
    images, labels, _meta = next(iter(loaders.pv_train))
    print("One train batch image tensor shape:", tuple(images.shape), "labels:", tuple(labels.shape))

## BaselineCNN architecture

Three **Conv → BatchNorm → ReLU → MaxPool** blocks, **adaptive** average pool to 4×4, then a small **MLP** head. Final selected run uses **dropout = 0.3** in the head. **Checkpoint:** `outputs/checkpoints/baseline_cnn_best_run3.pt` (**LR=3e-4**, **25** epochs, **weight_decay=1e-4**, **batch=32**).

In [ ]:
from models.cnn_baseline import BaselineCNN

m = BaselineCNN(num_classes=8, dropout=0.3)
n_params = sum(p.numel() for p in m.parameters())
print(f"BaselineCNN (dropout=0.3) — total parameters: {n_params:,}")
print(m)

## Training and evaluation framework (`trainer.py`)

Shared helpers: **`train_one_epoch`**, **`evaluate`**, **`run_training_loop`**, **`save_checkpoint`**. The **best** Baseline run is the one with the **lowest PlantVillage validation loss** (not PlantDoc). This notebook does **not** re-run long training; it only **displays** saved **JSON** and **PNGs** from prior runs. Optional entrypoint: `src/training/evaluate_final.py` (loads **run3** and **resnet18_best** checkpoints for final PV + PlantDoc evaluation).

## BaselineCNN tuning and final model (run3)

Hyperparameter runs used **only** **PlantVillage validation** to compare settings. **PlantDoc** was **not** part of that decision process.

- **Run 1** — lower LR, longer training improved validation vs “original” recipe.
- **Run 2** — weight decay further improved.
- **Run 3 (selected)** — **dropout=0.3** achieved the **lowest** validation loss → **only** this checkpoint is the project’s final Baseline.
- **Run 4 (rejected)** — stronger `ColorJitter` **worsened** validation and was not selected.

Below: table from saved JSON, plus the **run3** training-curve figure.

In [ ]:
TUNING = [
    ("Original", REPO_ROOT / "outputs/results/baseline_results.json", "15 ep, lr=1e-3, dropout=0.5, wd=0 (pre–run#)"),
    ("run1", REPO_ROOT / "outputs/results/baseline_results_run1.json", "lr=3e-4, 25 ep"),
    ("run2", REPO_ROOT / "outputs/results/baseline_results_run2.json", "+ weight_decay=1e-4"),
    ("run3 (selected)", REPO_ROOT / "outputs/results/baseline_results_run3.json", "+ dropout=0.3 — best val loss"),
    ("run4 (rejected)", REPO_ROOT / "outputs/results/baseline_results_run4.json", "stronger ColorJitter — worse val"),
]
rows = []
for run_name, path, change in TUNING:
    j = load_json_path(path)
    if not j:
        rows.append({"run": run_name, "change": change, "best_val_loss": None, "best_epoch": None, "pv_test_acc": None, "pv_test_f1": None})
    else:
        rows.append({
            "run": run_name,
            "change": change,
            "best_val_loss": j.get("best_val_loss"),
            "best_epoch": j.get("best_epoch"),
            "pv_test_acc": j.get("test_acc"),
            "pv_test_f1": j.get("test_f1"),
        })
tune_df = pd.DataFrame(rows)
if tune_df["best_val_loss"].notna().all():
    tune_df = tune_df.round({"best_val_loss": 4, "pv_test_acc": 4, "pv_test_f1": 4})
display(tune_df)
print("Final Baseline: outputs/checkpoints/baseline_cnn_best_run3.pt")
show_figure("outputs/figures/baseline_training_curves_run3.png")

## ResNet-18: architecture and fine-tuning setup

`build_resnet18_finetune(num_classes=8)` uses **IMAGENET1K_V1** weights, replaces **fc** with **512 → 8**, and **fully fine-tunes** all layers. **Why compare?** It tests **transfer learning** (generic visual features) against **from-scratch** Baseline on the same pipeline.

In [ ]:
from models.resnet18_finetune import build_resnet18_finetune

r = build_resnet18_finetune(8)
print("Total parameters:", f"{sum(p.numel() for p in r.parameters()):,}")
print("Final fc layer:", r.fc)

## ResNet-18 on PlantVillage (saved training run)

We load `outputs/results/resnet18_results.json` and the saved **training** curve and **end-of-training** PlantVillage confusion matrix. **Best val loss = 0.0103** at **epoch 12**; **PlantVillage test acc = 0.9989**, **macro-F1 = 0.9987** — no extra in-notebook fine-tuning was **needed** for near-ceiling in-domain performance in this run. **Checkpoint:** `outputs/checkpoints/resnet18_best.pt`.

In [ ]:
rjson = load_json_path(REPO_ROOT / "outputs/results/resnet18_results.json")
if rjson:
    print(f"best_val_loss = {rjson.get('best_val_loss'):.4f}  best_epoch = {rjson.get('best_epoch')}")
    print(f"PV test acc = {rjson.get('test_acc'):.4f}  macro-F1 = {rjson.get('test_f1'):.4f}")
else:
    print("WARNING: resnet18_results.json missing; expected: best val loss 0.0103, epoch 12, acc 0.9989, F1 0.9987")
show_figure("outputs/figures/resnet18_training_curves.png")
show_figure("outputs/figures/resnet18_confusion_matrix.png")

## Final evaluation: selected models on PV test and PlantDoc

`src/training/evaluate_final.py` loads **run3** and **resnet18_best** and evaluates with **one** shared eval transform. This section shows **`baseline_final_eval.json`**, **`resnet18_final_eval.json`**, and **`final_comparison.json`**, and the **official** Step-13 confusion figures.

**Key conclusion:** ResNet-18 **improves** **PlantDoc** **accuracy / F1** and **reduces** the **gap** vs Baseline, but **both** show a **large** **PlantVillage → PlantDoc** drop.

In [ ]:
fc = load_json_path(REPO_ROOT / "outputs/results/final_comparison.json")
if isinstance(fc, list) and len(fc) >= 2:
    out = []
    for row in fc:
        label = "Baseline run3" if row.get("model") == "baseline" else "ResNet-18"
        out.append({
            "model": label,
            "PV test acc": round(float(row["pv_test_acc"]), 4),
            "PlantDoc acc": round(float(row["plantdoc_acc"]), 4),
            "Accuracy gap": round(float(row["gap_acc"]), 4),
            "PV F1": round(float(row["pv_test_f1"]), 4),
            "PlantDoc F1": round(float(row["plantdoc_f1"]), 4),
            "F1 gap": round(float(row["gap_f1"]), 4),
        })
    display(pd.DataFrame(out))
else:
    print("WARNING: final_comparison.json missing — showing documented final aggregates.")
    display(pd.DataFrame([
        {"model": "Baseline run3", "PV test acc": 0.9718, "PlantDoc acc": 0.2330, "Accuracy gap": 0.7388, "PV F1": 0.9698, "PlantDoc F1": 0.1865, "F1 gap": 0.7833},
        {"model": "ResNet-18", "PV test acc": 0.9989, "PlantDoc acc": 0.4000, "Accuracy gap": 0.5989, "PV F1": 0.9987, "PlantDoc F1": 0.3202, "F1 gap": 0.6785},
    ]))

for rel in [
    "outputs/figures/baseline_pv_test_confusion_matrix.png",
    "outputs/figures/baseline_plantdoc_confusion_matrix.png",
    "outputs/figures/resnet18_pv_test_confusion_matrix.png",
    "outputs/figures/resnet18_plantdoc_confusion_matrix.png",
]:
    display(Markdown(f"`{rel}`"))
    show_figure(rel, width=640)

## Tuning and ablation (protocol reminder)

**All** decisions about LR, training length, weight decay, dropout, and the strong **ColorJitter** experiment were made using **only** **PlantVillage validation** (and the training set for learning). **PlantDoc** was **not** used to choose settings; we do **not** claim a knob was selected because it improved PlantDoc. **Run 4** was rejected because it **hurt** **PlantVillage** validation, not because of any PlantDoc result.

## Shortcut learning and failure-mode discussion (qualitative / aggregate)

- **Gaps (accuracy):** Baseline **0.7388** vs ResNet-18 **0.5989** (PV test minus PlantDoc) — Baseline **degrades more**; ResNet-18 still shows a **large** external gap.
- **Interpretation (careful):** Large gaps are **consistent with** **dataset shift**; off-diagonal errors in the PlantDoc CMs are **suggestive** of confusions (e.g. small tomato lesions, **cross-species** mixes) and **may** reflect **sensitivity to background, style, and lighting** — **not** a proof of a single spurious cue.
- **Per-class numbers:** this project’s JSON stores **aggregate** metrics; **exact** per-class confusion counts are **only** in the **PNG** figures here unless you export them separately — treat class-level **counts** as **visual/qualitative** in this report.

## Conclusion and future work

- **High PlantVillage test accuracy is not** enough to assert **reliable** performance on different imaging conditions (e.g. PlantDoc).
- **ResNet-18** attains **better** **external** performance than the final Baseline, **plausibly** because of **pretrained** representations; the **from-scratch** model has a **larger** **gap**.
- **Both** models need **serious** **external** validation before field deployment claims.
- **Future work:** more **field** training data, **domain adaptation** / self-supervised pretraining, **tighter crops** (segmentation), more careful augmentation search, and **other** **external** datasets to stress-test generalization.

In [ ]:
# Optional: EDA text summary (if committed under outputs/results)
p_eda = REPO_ROOT / "outputs/results/final_subset_eda_summary.md"
if p_eda.exists():
    print(p_eda.read_text(encoding="utf-8")[:4000])
else:
    print("NOTE: final_subset_eda_summary.md not found — see docs/FINAL_CLASS_SUBSET.md for counts.")

---
**Artifacts this notebook uses:** `configs/class_subset_v1.json`; `outputs/results/baseline_results*.json`, `resnet18_results.json`, `*_final_eval.json`, `final_comparison.json`; figures under `outputs/figures/*.png` as shown; **checkpoints** `outputs/checkpoints/baseline_cnn_best_run3.pt` and `resnet18_best.pt` (not `torch.load`’d here by default). **Re-run final eval (optional):** `python src/training/evaluate_final.py` from repo root with data paths configured.